# Complete the linkage between scotus & circuit court opinions using the data from 2a and 2c, using scotus scraped data as bridge

# Import Libraries

In [44]:
import json

import numpy as np
import pandas as pd

# Load the data from 2a and 2c

In [3]:
joined_scotus = pd.read_csv("data/joined_scotus.csv")
len(joined_scotus)

172045

In [4]:
joined_scotus.columns

Index(['cl_docket_id', 'cl_docket_number', 'cl_cluster_id', 'cl_case_name',
       'cl_date_filed', 'cl_precedential_status', 'cl_num_authorities',
       'cl_cleaned_docket_number', 'cl_year_filed', 'cl_year_bucket',
       'cl_docket_type', 'cl_docket_year', 'scotus_docket_number',
       'scotus_filename', 'scotus_docket_date', 'scotus_case_title',
       'scotus_lower_court', 'scotus_lower_court_case_numbers_raw',
       'scotus_lower_court_case_numbers', 'scotus_lower_court_decision_date',
       'scotus_lower_court_rehearing_denied_date', 'scotus_year',
       'scotus_case_num'],
      dtype='object')

In [5]:
joined_circuit = pd.read_csv("data/joined_circuit.csv")
len(joined_circuit)

/var/folders/d9/3h7m7wc52kv6fgxmbyd8s0940000gn/T/ipykernel_14122/1097889423.py:1: DtypeWarning: Columns (8) have mixed types. Specify dtype option on import or set low_memory=False.
  joined_circuit = pd.read_csv("data/joined_circuit.csv")


114650

In [6]:
joined_circuit.columns

Index(['scotus_docket_number', 'scotus_filename', 'scotus_docket_date',
       'scotus_case_title', 'scotus_lower_court',
       'scotus_lower_court_case_numbers_raw',
       'scotus_lower_court_case_numbers', 'scotus_lower_court_decision_date',
       'scotus_lower_court_rehearing_denied_date', 'scotus_year',
       'scotus_case_num', 'scotus_lower_court_case_number',
       'cl_lower_court_docket_id', 'cl_lower_court_docket_number',
       'cl_lower_court_cluster_id', 'cl_lower_court_case_name',
       'cl_lower_court_date_filed', 'cl_lower_court_precedential_status',
       'cl_lower_court_id', 'cl_lower_court_docket_number_clean',
       'low_ngram_overlap', 'cl_lower_court_name', 'cl_lower_court_type'],
      dtype='object')

# Join the two dfs on scotus columns

In [8]:
join_cols = list(set(joined_scotus.columns) & set(joined_circuit.columns))
join_cols

['scotus_lower_court',
 'scotus_lower_court_case_numbers_raw',
 'scotus_lower_court_case_numbers',
 'scotus_docket_number',
 'scotus_lower_court_decision_date',
 'scotus_filename',
 'scotus_lower_court_rehearing_denied_date',
 'scotus_case_num',
 'scotus_year',
 'scotus_case_title',
 'scotus_docket_date']

In [9]:
joined_df = pd.merge(joined_scotus, joined_circuit, how="left", on=join_cols)
len(joined_df)

202456

In [28]:
joined_df = joined_df[~joined_df["cl_lower_court_docket_number"].isnull()]
joined_df["cl_lower_court_cluster_id"] = joined_df["cl_lower_court_cluster_id"].astype(int)
len(joined_df)

108054

In [10]:
joined_df.columns

Index(['cl_docket_id', 'cl_docket_number', 'cl_cluster_id', 'cl_case_name',
       'cl_date_filed', 'cl_precedential_status', 'cl_num_authorities',
       'cl_cleaned_docket_number', 'cl_year_filed', 'cl_year_bucket',
       'cl_docket_type', 'cl_docket_year', 'scotus_docket_number',
       'scotus_filename', 'scotus_docket_date', 'scotus_case_title',
       'scotus_lower_court', 'scotus_lower_court_case_numbers_raw',
       'scotus_lower_court_case_numbers', 'scotus_lower_court_decision_date',
       'scotus_lower_court_rehearing_denied_date', 'scotus_year',
       'scotus_case_num', 'scotus_lower_court_case_number',
       'cl_lower_court_docket_id', 'cl_lower_court_docket_number',
       'cl_lower_court_cluster_id', 'cl_lower_court_case_name',
       'cl_lower_court_date_filed', 'cl_lower_court_precedential_status',
       'cl_lower_court_id', 'cl_lower_court_docket_number_clean',
       'low_ngram_overlap', 'cl_lower_court_name', 'cl_lower_court_type'],
      dtype='object')

In [18]:
assert len(joined_df[joined_df["cl_cleaned_docket_number"] != joined_df["scotus_docket_number"]]) == 0

In [19]:
assert len(joined_df[joined_df["scotus_lower_court_case_number"] != joined_df["cl_lower_court_docket_number_clean"]]) == 0

In [20]:
joined_df["cl_cluster_id"].value_counts()

cl_cluster_id
9280505    626
9281142     93
8433175     90
2959748     84
9063594     72
          ... 
8419894      1
8420160      1
8419870      1
8419808      1
9367251      1
Name: count, Length: 74125, dtype: int64

In [29]:
joined_df[joined_df["cl_cluster_id"] == 9280505][["cl_cluster_id", "cl_cleaned_docket_number", "scotus_lower_court", "scotus_lower_court_case_number", "cl_lower_court_cluster_id"]]

,cl_cluster_id,cl_cleaned_docket_number,scotus_lower_court,scotus_lower_court_case_number,cl_lower_court_cluster_id
181164,9280505,03-9262,United States Court of Appeals for the Fifth C...,02-41157,39543
181165,9280505,03-9262,United States Court of Appeals for the Fifth C...,02-41157,33456
181166,9280505,03-10705,United States Court of Appeals for the Distric...,02-3043,186160
181167,9280505,03-10727,United States Court of Appeals for the Ninth C...,03-50098,791434
181168,9280505,03-10727,United States Court of Appeals for the Ninth C...,03-50098,3032955
...,...,...,...,...,...
181822,9280505,04-7793,United States Court of Appeals for the Fifth C...,03-60832,41494
181823,9280505,04-7816,United States Court of Appeals for the Elevent...,03-16388,76906
181825,9280505,04-7827,United States Court of Appeals for the Elevent...,04-12515,43864
181827,9280505,04-7848,United States Court of Appeals for the Ninth C...,03-50597,8456459


In [30]:
joined_df[joined_df["cl_cluster_id"] == 9063594][["cl_cluster_id", "cl_cleaned_docket_number", "scotus_lower_court", "scotus_lower_court_case_number", "cl_lower_court_cluster_id"]]

,cl_cluster_id,cl_cleaned_docket_number,scotus_lower_court,scotus_lower_court_case_number,cl_lower_court_cluster_id
95527,9063594,11-5323,United States Court of Appeals for the Seventh...,11-1202,3004447
95529,9063594,11-5950,United States Court of Appeals for the Seventh...,10-3360,213669
95530,9063594,11-5950,United States Court of Appeals for the Seventh...,10-3360,8466211
95531,9063594,11-6364,United States Court of Appeals for the Seventh...,10-3450,806878
95532,9063594,11-6364,United States Court of Appeals for the Seventh...,10-3450,8691340
...,...,...,...,...,...
95602,9063594,11-9711,United States Court of Appeals for the Fifth C...,10-30920,619103
95603,9063594,11-9938,United States Court of Appeals for the Fifth C...,10-31022,806712
95604,9063594,11-9938,United States Court of Appeals for the Fifth C...,10-31022,733779
95605,9063594,11-9961,United States Court of Appeals for the Fifth C...,11-30366,733749


# Make the link between CL scotus cluster_id to CL circuit cluster_id

In [31]:
joined_df["cl_cluster_id"].nunique()

74125

In [40]:
joined_df["cl_cleaned_docket_number"].nunique()

47439

In [42]:
len(joined_df[["scotus_lower_court", "scotus_lower_court_case_number"]].drop_duplicates())

49815

In [43]:
joined_df["cl_lower_court_cluster_id"].nunique()

56514

In [36]:
cluster_id_map = joined_df.groupby("cl_cluster_id")["cl_lower_court_cluster_id"].apply(list).to_dict()
assert joined_df["cl_cluster_id"].nunique() == len(cluster_id_map.keys())

# Save the data

In [45]:
with open("data/scotus_to_circuit_cluster_id_map.json", "w") as f:
    json.dump(cluster_id_map, f)